# 02 — Evaluate a dataset with metrics

The model layer + metrics + the evaluate engine wired together.

**Needs a live Azure OpenAI model** and the model dependencies
(`langchain` / `ragas` / …) for every metric that calls an LLM.

Set these first:

```
LLMINSPECTOR_AZURE_ENDPOINT
LLMINSPECTOR_API_VERSION
LLMINSPECTOR_API_KEY
```

(or pass an `azure_ad_token_provider` in code instead).

## 1. Configure the provider

`AzureSettings.from_env()` reads the variables above. In code you can pass
`api_key` **or** an `azure_ad_token_provider` explicitly — exactly one.

In [ ]:
from llminspector.config import AzureSettings
from llminspector.models import AzureOpenAIModel

settings = AzureSettings.from_env()
model = AzureOpenAIModel(settings)   # picks up settings.api_key
model.get_model_name()

## 2. Build the metric set

Each metric is an object constructed with the model it needs. Local-only metrics
(BERTScore, PII) take no model.

In [ ]:
from llminspector.metrics import (
    AnswerCorrectnessMetric,
    BertScoreMetric,
    FaithfulnessMetric,
    PIIDetectionMetric,
    SentimentMetric,
)

metrics = [
    FaithfulnessMetric(model),
    AnswerCorrectnessMetric(model),   # unified judge: GT + faithfulness + relevancy
    SentimentMetric(model, target="actual_output"),
    BertScoreMetric(),
    PIIDetectionMetric(target="actual_output"),
]
[m.name for m in metrics]

## 3. The dataset under test

Inline here; normally `EvaluationDataset.from_excel(...)`.

In [ ]:
from llminspector.dataset import EvaluationDataset
from llminspector.test_case import LLMTestCase

dataset = EvaluationDataset(
    test_cases=[
        LLMTestCase(
            input="What is the capital of France?",
            actual_output="Paris is the capital of France.",
            expected_output="Paris",
            retrieval_context=["The capital of France is Paris."],
        ),
    ]
)
dataset.to_pandas()

## 4. Evaluate

Jupyter already runs an event loop, so **await the async entry point**. The sync
`evaluate(...)` is for scripts and raises here.

Metrics whose required inputs are missing on a row are skipped for that row.

In [ ]:
from llminspector import a_evaluate

result = await a_evaluate(dataset, metrics)
result.to_pandas().T

## 5. Report

`summary()` gives per-metric numeric stats. `errors()` is where a failed metric
shows up — without it, a failure and a skipped row both just look like `None`.

In [ ]:
from llminspector import reporting

print(reporting.summary(result))
reporting.errors(result)

In [ ]:
result.to_excel("/tmp/llminspector_eval.xlsx")
print("Wrote /tmp/llminspector_eval.xlsx")